# 14 Organization-Wide Skill Gap Heatmap
**Enterprise HR AI — Workforce Intelligence & Upskilling Platform**

### Purpose:
Aggregate missing skill frequencies across departments and assign transparent severity thresholds.


In [2]:
import os
import pandas as pd
from collections import Counter

DATA_PROCESSED = "../data/processed"
df_gaps = pd.read_csv(os.path.join(DATA_PROCESSED, "employee_skill_gaps.csv"))
df_attrition = pd.read_csv(os.path.join(DATA_PROCESSED, "employee_attrition_processed.csv"))

df_merged = pd.merge(df_gaps, df_attrition[['employee_id', 'department']], on='employee_id', how='inner')
total_workforce = len(df_merged)

# Org-level gap frequencies
all_missing = []
for missing_str in df_merged['missing_skills_list'].dropna():
    if missing_str:
        all_missing.extend(missing_str.split('|'))

gap_counts = Counter(all_missing)
org_gap_records = []

for skill, count in gap_counts.most_common():
    pct = round((count / total_workforce) * 100, 2)
    # Severity rule: >= 30% missing -> HIGH, 15-29% -> MEDIUM, < 15% -> LOW
    if pct >= 30.0:
        severity = "HIGH"
    elif pct >= 15.0:
        severity = "MEDIUM"
    else:
        severity = "LOW"
        
    org_gap_records.append({
        'skill_name': skill,
        'employees_missing_count': count,
        'missing_percentage': pct,
        'severity_level': severity
    })

df_org_gaps = pd.DataFrame(org_gap_records)
out_org_gaps = os.path.join(DATA_PROCESSED, "organization_skill_gaps.csv")
df_org_gaps.to_csv(out_org_gaps, index=False)

print("=== TOP 15 CRITICAL ORGANIZATIONAL SKILL GAPS ===")
print(df_org_gaps.head(15).to_string(index=False))


=== TOP 15 CRITICAL ORGANIZATIONAL SKILL GAPS ===
                   skill_name  employees_missing_count  missing_percentage severity_level
        Reading Comprehension                      754               51.29           HIGH
             Active Listening                      745               50.68           HIGH
                      Writing                      742               50.48           HIGH
            Critical Thinking                      738               50.20           HIGH
              Active Learning                      728               49.52           HIGH
                   Monitoring                      725               49.32           HIGH
                     Speaking                      718               48.84           HIGH
              Microsoft Excel                      589               40.07           HIGH
    Microsoft Office software                      559               38.03           HIGH
         Microsoft PowerPoint                     